# Notebook 04 — Scalable Feature Engineering
**MSBDA-801 | Arabic AI Text Detection**

**Phase 3 — Tasks 3.1 / 3.2 / 3.3**

## Assigned features — formula  f_((k×n)+i),  n=21

| Student | k=0 | k=1 | k=2 | k=3 | k=4 | k=5 |
|---------|-----|-----|-----|-----|-----|-----|
| i=1     | f1  | f22 | f43 | f64 | f85 | f106 |
| i=4     | f4  | f25 | f46 | f67 | f88 | f109 |
| i=13    | f13 | f34 | f55 | f76 | f97 | —    |

**`FORCE_REBUILD_FEATURES`**: Default `False`. Set `True` only after
changing `src/feature_engineering.py`.

In [1]:
# ── Session bootstrap (run at the top of every new Colab session) ─────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, importlib
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection')
SRC_DIR = PROJECT_ROOT / 'src'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# src/__init__.py is committed to the repo, no runtime touch needed
print('PROJECT_ROOT:', PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


In [15]:
!pip install -q camel-tools >=1.5.2

In [2]:
import shutil, importlib
import src.feature_engineering as fe_mod
importlib.reload(fe_mod)

from src.utils import (
    create_spark_session, load_checkpoint, save_checkpoint,
    checkpoint_exists, add_src_to_spark, Timer,
    FIGURES_DIR, MODELS_DIR, DATA_PROCESSED, FEAT_PREFIX, CAMEL_CACHE,
)
from src.feature_engineering import (
    extract_all_features, build_tfidf_pipeline, build_word2vec_pipeline,
    assemble_feature_vector, FEATURE_COLS, print_mapreduce_design,
    _FEATURE_REGISTRY,
)
from src.data_preparation import stratified_split
from pyspark.sql import functions as F
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

os.environ["CAMELTOOLS_DATA"] = str(CAMEL_CACHE)
print("CAMELTOOLS_DATA:", os.environ["CAMELTOOLS_DATA"])
print("Cache exists:", CAMEL_CACHE.exists())

spark = create_spark_session('ArabicAIDetection_Features')
add_src_to_spark(spark)


CAMELTOOLS_DATA: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/.camel_cache
Cache exists: True


'/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/src_package.zip'

## 1. Optionally clean old feature checkpoints

In [22]:
FORCE_REBUILD_FEATURES = False   # ← set True after changing feature_engineering.py

FEATURE_CHECKPOINTS = [
    'features_stylometric', 'features_tfidf', 'features_w2v',
    'split_train', 'split_val', 'split_test',
    'split_train_assembled', 'split_val_assembled', 'split_test_assembled',
]
MODEL_CHECKPOINTS = ['tfidf_pipeline', 'w2v_pipeline']

if FORCE_REBUILD_FEATURES:
    for name in FEATURE_CHECKPOINTS:
        p = DATA_PROCESSED / name
        if p.exists():
            shutil.rmtree(p)
            print('Deleted:', p)
    for name in MODEL_CHECKPOINTS:
        p = MODELS_DIR / name
        if p.exists():
            shutil.rmtree(p)
            print('Deleted model:', p)


Deleted: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/data/processed/features_stylometric


## 2. Show feature registry

In [3]:
print('Feature columns to be computed:')
for key, (_, rtype, col_name) in _FEATURE_REGISTRY.items():
    print(f'  {col_name:50s}  key={key:28s}  type={rtype}')
print(f'\nTotal: {len(FEATURE_COLS)} stylometric features')


Feature columns to be computed:
  feat_f1_total_chars                                 key=f1_total_chars                type=IntegerType()
  feat_f22_word_entropy                               key=f22_word_entropy              type=DoubleType()
  feat_f43_num_nouns                                  key=f43_num_nouns                 type=IntegerType()
  feat_f64_num_nominatives                            key=f64_num_nominatives           type=IntegerType()
  feat_f85_sent_len_variance                          key=f85_sent_len_variance         type=DoubleType()
  feat_f106_tanween_freq                              key=f106_tanween_freq             type=IntegerType()
  feat_f4_whitespace_ratio                            key=f4_whitespace_ratio           type=DoubleType()
  feat_f25_single_quotes                              key=f25_single_quotes             type=IntegerType()
  feat_f46_num_adverbs                                key=f46_num_adverbs               type=IntegerType()
  feat_f

## 3. Load preprocessed data

In [4]:
preprocessed_df = load_checkpoint(spark, 'preprocessed')
preprocessed_df.cache()
print(f'Loaded {preprocessed_df.count():,} rows')
preprocessed_df.select('label', 'source_model', 'generation_method', 'clean_text').show(5, truncate=80)


Loaded 41,940 rows
+-----+------------+-----------------+--------------------------------------------------------------------------------+
|label|source_model|generation_method|                                                                      clean_text|
+-----+------------+-----------------+--------------------------------------------------------------------------------+
|    1|      openai|     by_polishing|صور نظام تعليم مراه اندلسيه تستند دراسه دقيقه مصادر تاريخيه وثقت حياه علماء م...|
|    1|      openai|     by_polishing|انهيار دوله موحد يعود بشكل كبير عوامل ثقافيه، تقل اهميه عوامل سياسيه عسكريه م...|
|    1|      openai|     by_polishing|جهود قاده ثوره جزائريه مرحله اولي حاسمه بحث مصادر تسليح عبر حدود غربيه، اسهمت...|
|    1|      openai|     by_polishing|مقال يناقش قضيه ضرائب شرعيه دولتي مرابط موحدين، موضحا تاثير ضرائب علاقه دوله ...|
|    1|      openai|     by_polishing|حركه انتصار حري ديمقراطيه جزائر، فتره شهدت تحول جذريه بدات حرب عالميه ثانيه و...|
+-----+------------+-

## 4. MapReduce design note (for Methodology section)

In [5]:
print_mapreduce_design()



══════════════════════════════════════════════════════════════
  MapReduce Design — Hapax Legomena Ratio (f13)
══════════════════════════════════════════════════════════════

Job 1 — Word Count
  MAP:    (doc_id, text)  →  (word, 1)  for each word
  REDUCE: (word, [1,1,…]) →  (word, total_count)

Job 2 — Hapax Identification
  MAP:    (word, total_count) | filter count==1
                              → ("HAPAX_KEY", 1)
  REDUCE: ("HAPAX_KEY", [1,1,…]) → hapax_total

Final Calculation
  hapax_ratio = hapax_total / total_word_count

Spark equivalent:
  Job1 → df.explode + groupBy("word").count()
  Job2 → filter(count == 1).count()
  Final → hapax_count / total_count
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  MapReduce Design — Word Entropy (f22)
══════════════════════════════════════════════════════════════

Job 1 — Word Count  (same as above)

Job 2 — Entropy Calculation
  MAP:    (word, count) → p =

## 5. Extract all 17 stylometric features (Task 3.1)

If CAMeL Tools or transformers are unavailable, morphology and
embedding features safely default to 0 per the feature functions.

In [6]:
if checkpoint_exists('features_stylometric'):
    features_df = load_checkpoint(spark, 'features_stylometric')
    print('Loaded stylometric features checkpoint.')
else:
    with Timer('Extract 17 stylometric features'):
        features_df = extract_all_features(preprocessed_df, input_col='clean_text')
        save_checkpoint(features_df, 'features_stylometric')
    print('Stylometric features extracted and saved.')

features_df.cache()
print(f'Rows: {features_df.count():,}')

available = [c for c in features_df.columns if c.startswith(FEAT_PREFIX)]
missing   = [c for c in FEATURE_COLS if c not in features_df.columns]
print(f'Available feature columns: {len(available)}')
if missing:
    print('WARNING — missing columns:', missing)
else:
    print(' All 17 feature columns present.')


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 6. Build TF-IDF features (Task 3.2)

In [7]:
if checkpoint_exists('features_tfidf'):
    tfidf_df = load_checkpoint(spark, 'features_tfidf')
    print('Loaded TF-IDF checkpoint.')
else:
    tfidf_pipeline = build_tfidf_pipeline(
        num_features=20_000, min_df=2,
        input_col='clean_text', output_col='tfidf_features',
    )
    with Timer('TF-IDF fit + transform'):
        tfidf_model = tfidf_pipeline.fit(features_df)
        tfidf_df    = tfidf_model.transform(features_df).drop('_tokens', '_tf_raw')
        save_checkpoint(tfidf_df, 'features_tfidf')

    tfidf_model.write().overwrite().save(str(MODELS_DIR / 'tfidf_pipeline'))
    print('TF-IDF pipeline saved to GDrive.')

print('tfidf_features column present:', 'tfidf_features' in tfidf_df.columns)
print(f'Rows: {tfidf_df.count():,}')


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 7. Optional — Word2Vec embeddings (Task 3.2)

In [ ]:
RUN_WORD2VEC = False   # ← set True to include Word2Vec in modeling

if RUN_WORD2VEC:
    if checkpoint_exists('features_w2v'):
        w2v_df = load_checkpoint(spark, 'features_w2v')
        print('Loaded Word2Vec checkpoint.')
    else:
        w2v_pipeline = build_word2vec_pipeline(vector_size=100, min_count=2)
        with Timer('Word2Vec fit + transform'):
            w2v_model = w2v_pipeline.fit(tfidf_df)
            w2v_df    = w2v_model.transform(tfidf_df)
            save_checkpoint(w2v_df, 'features_w2v')
        w2v_model.write().overwrite().save(str(MODELS_DIR / 'w2v_pipeline'))
        print('Word2Vec pipeline saved.')
else:
    print('Word2Vec skipped (RUN_WORD2VEC=False).')


## 8. Stratified train / validation / test split (Task 3.3)

In [ ]:
if (checkpoint_exists('split_train') and checkpoint_exists('split_val')
        and checkpoint_exists('split_test')):
    train_df = load_checkpoint(spark, 'split_train')
    val_df   = load_checkpoint(spark, 'split_val')
    test_df  = load_checkpoint(spark, 'split_test')
    print('Loaded split checkpoints.')
else:
    train_df, val_df, test_df = stratified_split(
        tfidf_df, train_ratio=0.70, val_ratio=0.15, seed=42
    )
    save_checkpoint(train_df, 'split_train')
    save_checkpoint(val_df,   'split_val')
    save_checkpoint(test_df,  'split_test')
    print('Splits saved.')

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n   = df.count()
    pos = df.filter(F.col('label') == 1).count()
    print(f'{name:5s}: {n:6,} rows  AI={pos/n*100:.1f}%')


## 9. Assemble final feature vector

In [ ]:
for split_name, df_in in [
    ('split_train_assembled', train_df),
    ('split_val_assembled',   val_df),
    ('split_test_assembled',  test_df),
]:
    if checkpoint_exists(split_name):
        print(f'{split_name}: already exists, skipping.')
    else:
        with Timer(f'Assemble {split_name}'):
            assembled = assemble_feature_vector(df_in, tfidf_col='tfidf_features')
            save_checkpoint(assembled, split_name)
        print(f'{split_name}: assembled and saved.')


## 10. Feature statistics

In [ ]:
assembled_train = load_checkpoint(spark, 'split_train_assembled')
print('features column present:', 'features' in assembled_train.columns)

feat_stats = (
    assembled_train
    .select([F.col(c).cast('double').alias(c) for c in FEATURE_COLS if c in assembled_train.columns])
    .describe()
)
feat_stats.show(truncate=False)
